# CXR Sentinel — Phase 1 (Colab)

Core pipeline: 3-finding classifier (cardiomegaly, pleural effusion, lung opacity) + Grad-CAM + calibration.

**Run this notebook top to bottom once, with the synthetic self-test, before you have real data downloaded.**
It proves the whole pipeline (data loading -> training -> Grad-CAM -> calibration) is wired correctly, so when
your CheXpert Plus / MIMIC-CXR / NIH ChestX-ray14 access comes through you're only swapping in a CSV + image
folder, not debugging code for the first time.

Runtime: use a GPU runtime (Runtime -> Change runtime type -> T4 GPU). Free tier is enough for Phase 1 at a
few thousand images.

## 0. Setup

In [ ]:
# Option A: clone your own repo (push this scaffold to GitHub first, then swap in your URL)
# !git clone https://github.com/<your-username>/cxr-sentinel.git
# %cd cxr-sentinel

# Option B: if you uploaded the cxr-sentinel/ folder directly into the Colab file browser
# or into Google Drive, just cd into it instead:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/cxr-sentinel

import os
print("Working dir:", os.getcwd())
print("Contents:", os.listdir('.'))

In [ ]:
!pip install -q -r requirements.txt

## 1. Smoke test on synthetic data

This creates fake random images + a fake label CSV, runs the full pipeline end to end, and asserts
every stage produces the right shapes. If this passes, the code is correct and any problems you hit
later are data problems, not pipeline problems.

In [ ]:
!python -m src.selftest

## 2. Point at real data

You need a CSV with these columns (see `src/data.py` docstring):

```
image_path, patient_id, study_id, study_date, cardiomegaly, pleural_effusion, lung_opacity
```

**Getting data, starting today:**
- **NIH ChestX-ray14** — no credentialing required. Run `!python -m scripts.download_nih_chestxray14 --num_batches 12`
  in the next cell (full dataset, do this on Colab's disk/bandwidth, not locally). Already writes `data/train.csv` /
  `data/val.csv` in this repo's schema. **Caveat:** NIH has no native "Lung Opacity" label; the script maps its
  "Infiltration" label to `lung_opacity` as a placeholder — see the script's docstring.
- **CheXpert Plus** (Stanford AIMI) — apply now, no CITI training needed, usually faster to clear. Has multiple
  studies per patient, which you'll need for Phase 2 (longitudinal).
- **MIMIC-CXR** (PhysioNet) — apply in parallel; needs a completed CITI human-subjects course + signed data use
  agreement, budget 1-2 weeks. Also has repeat studies per patient, plus free-text reports for Phase 3 (VLM).

Both CheXpert Plus and MIMIC-CXR are research-use-only licenses — not for commercial redistribution. If this is
headed toward an actual product rather than a research prototype, that licensing question (and the FDA
Software-as-a-Medical-Device angle for anything that outputs findings a clinician could act on) needs an answer
before you scale past a demo — worth a quick conversation with whoever handles that at your company.

In [ ]:
# Full dataset (~42GB, 112,120 images) — fine on Colab's disk, run once:
!python -m scripts.download_nih_chestxray14 --num_batches 12

# Or grab fewer batches first to sanity-check on real images faster:
# !python -m scripts.download_nih_chestxray14 --num_batches 2

In [ ]:
import yaml
from src.data import CXRDataset, CXRDatasetConfig

with open('configs/phase1.yaml') as f:
    config = yaml.safe_load(f)

targets = config['targets']

# Fill these in once configs/phase1.yaml points at real files:
train_cfg = CXRDatasetConfig(
    csv_path=config['data']['train_csv'],
    image_root=config['data']['image_root'],
    targets=targets,
    image_size=config['data']['image_size'],
    train=True,
)
val_cfg = CXRDatasetConfig(
    csv_path=config['data']['val_csv'],
    image_root=config['data']['image_root'],
    targets=targets,
    image_size=config['data']['image_size'],
    train=False,
)

# Uncomment once data/train.csv and data/val.csv exist:
# train_ds = CXRDataset(train_cfg)
# val_ds = CXRDataset(val_cfg)
# print(len(train_ds), len(val_ds))

## 3. Train

In [ ]:
from src.train import run_training

# model, history = run_training(
#     train_ds, val_ds, targets,
#     output_dir='checkpoints',
#     epochs=config['train']['epochs'],
#     batch_size=config['train']['batch_size'],
#     lr=config['train']['lr'],
# )

## 4. Grad-CAM on a sample

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.gradcam import GradCAM

# sample_image, sample_labels, meta = val_ds[0]
# model.eval()
# cam = GradCAM(model)
# heatmap = cam(sample_image.unsqueeze(0), target_index=0)  # index 0 = first target in `targets`
#
# img_np = sample_image.permute(1, 2, 0).numpy()
# img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
#
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# axes[0].imshow(img_np); axes[0].set_title('Input'); axes[0].axis('off')
# axes[1].imshow(img_np); axes[1].imshow(heatmap, cmap='jet', alpha=0.4)
# axes[1].set_title(f'Grad-CAM: {targets[0]}'); axes[1].axis('off')
# plt.show()

## 5. Calibration + evaluation report

Run this from a terminal cell once you have a trained checkpoint:

In [ ]:
# !python -m src.evaluate --checkpoint checkpoints/best_model.pt \
#     --csv data/val.csv --image_root data/images --out_dir reports/

## Next: Phase 2 (longitudinal)

Once Phase 1's AUROC/calibration numbers look reasonable on real data:
1. Add `prior_study_id` pairing logic in a new `src/temporal.py` — join each study to the most recent prior
   study for the same `patient_id` from the CSV you already have.
2. Reuse `GradCAM` on both studies, compute an IoU / overlap score between the two heatmaps + a probability
   delta, and turn that into "worsening / improving / unchanged" language.
3. Don't build a custom temporal deep learning model yet — the delta-based approach above gets you a
   demoable longitudinal feature without a second research project.